# Extracting data from digikala

In [2]:
!uv add requests tqdm

Resolved 267 packages in 2ms
Checked 242 packages in 19ms


In [6]:
import json
import time
import requests
from pathlib import Path

BASE_URL = "https://api.digikala.com"
CATEGORY_URL = f"{BASE_URL}/v1/categories/mobile-phone/search/"

OUTPUT_DIR = Path("digikala_data")
OUTPUT_DIR.mkdir(exist_ok=True)

IDS_FILE = OUTPUT_DIR / "product_ids.json"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/140.0 Safari/537.36"
    ),
    "Accept": "application/json",
}


session = requests.Session()
session.headers.update(HEADERS)


def get_page(page):
    response = session.get(
        CATEGORY_URL,
        params={"page": page},
        timeout=60
    )

    response.raise_for_status()

    return response.json()


def get_all_product_ids():

    product_ids = set()

    if IDS_FILE.exists():
        with open(IDS_FILE, "r", encoding="utf-8") as f:
            product_ids = set(json.load(f))
        print(
            f"Loaded {len(product_ids)} existing product IDs."
        )

    data = get_page(1)

    pager = data["data"]["pager"]

    total_pages = pager["total_pages"]
    total_items = pager["total_items"]

    print(f"Total items: {total_items}")
    print(f"Total pages: {total_pages}")

    for page in range(1, total_pages + 1):

        print(
            f"\nGetting page {page}/{total_pages}..."
        )

        try:

            data = get_page(page)

            products = data["data"]["products"]

            page_ids = [
                product["id"]
                for product in products
                if product.get("id") is not None
            ]

            old_count = len(product_ids)

            product_ids.update(page_ids)

            print(
                f"Products in page: {len(page_ids)} | "
                f"New IDs: {len(product_ids) - old_count} | "
                f"Total IDs: {len(product_ids)}"
            )

            # ذخیره بعد از هر صفحه
            with open(
                IDS_FILE,
                "w",
                encoding="utf-8"
            ) as f:

                json.dump(
                    sorted(product_ids),
                    f,
                    ensure_ascii=False,
                    indent=2
                )

            time.sleep(1)

        except Exception as e:

            print(
                f"ERROR on page {page}: {e}"
            )

            print(
                "Stopping. Existing IDs have been saved."
            )

            break

    return product_ids

if __name__ == "__main__":

    ids = get_all_product_ids()

    print("\n" + "=" * 50)
    print(f"Final number of unique products: {len(ids)}")
    print(f"Saved to: {IDS_FILE.absolute()}")

Total items: 3955
Total pages: 198

Getting page 1/198...
Products in page: 20 | New IDs: 20 | Total IDs: 20

Getting page 2/198...
Products in page: 20 | New IDs: 20 | Total IDs: 40

Getting page 3/198...
Products in page: 20 | New IDs: 20 | Total IDs: 60

Getting page 4/198...
Products in page: 20 | New IDs: 20 | Total IDs: 80

Getting page 5/198...
Products in page: 20 | New IDs: 20 | Total IDs: 100

Getting page 6/198...
Products in page: 20 | New IDs: 20 | Total IDs: 120

Getting page 7/198...
Products in page: 20 | New IDs: 20 | Total IDs: 140

Getting page 8/198...
Products in page: 20 | New IDs: 20 | Total IDs: 160

Getting page 9/198...
Products in page: 20 | New IDs: 20 | Total IDs: 180

Getting page 10/198...
Products in page: 20 | New IDs: 20 | Total IDs: 200

Getting page 11/198...
Products in page: 20 | New IDs: 20 | Total IDs: 220

Getting page 12/198...
Products in page: 20 | New IDs: 20 | Total IDs: 240

Getting page 13/198...
Products in page: 20 | New IDs: 20 | Total

In [10]:
import json
import time
import requests
from pathlib import Path

INPUT_FILE = "digikala_data/product_ids.json"
OUTPUT_FILE = "digikala_data/products.jsonl"

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json"
}

session = requests.Session()
session.headers.update(HEADERS)


def get_product(product_id):
    url = f"https://api.digikala.com/v2/product/{product_id}/"

    response = session.get(
        url,
        timeout=60
    )

    response.raise_for_status()

    return response.json()


def extract_product(data):

    product = data["data"]["product"]

    # عنوان
    title = product.get("title_fa")

    # برند
    brand = (
        product
        .get("data_layer", {})
        .get("brand")
    )

    # قیمت
    price = None

    try:
        price = (
            product["default_variant"]
            ["price"]
            ["selling_price"]
        )
    except:
        pass

    # مشخصات
    specifications = {}

    for section in product.get(
        "specifications",
        []
    ):

        section_title = section.get(
            "title",
            "بدون دسته"
        )

        section_specs = {}

        for attr in section.get(
            "attributes",
            []
        ):

            attr_title = attr.get(
                "title"
            )

            values = []

            for value in attr.get(
                "values",
                []
            ):

                if isinstance(value, dict):
                    values.append(
                        value.get(
                            "title",
                            ""
                        )
                    )
                else:
                    values.append(
                        str(value)
                    )

            section_specs[
                attr_title
            ] = " | ".join(values)

        specifications[
            section_title
        ] = section_specs

    return {
        "id": product["id"],
        "title": title,
        "brand": brand,
        "price": price,
        "specifications": specifications
    }


def load_processed_ids():

    processed = set()

    if not Path(
        OUTPUT_FILE
    ).exists():

        return processed

    with open(
        OUTPUT_FILE,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            try:
                item = json.loads(line)
                processed.add(
                    item["id"]
                )
            except:
                pass

    return processed


def save_jsonl(item):

    with open(
        OUTPUT_FILE,
        "a",
        encoding="utf-8"
    ) as f:

        f.write(
            json.dumps(
                item,
                ensure_ascii=False
            )
            + "\n"
        )


def main():

    with open(
        INPUT_FILE,
        "r",
        encoding="utf-8"
    ) as f:

        product_ids = json.load(f)

    processed = load_processed_ids()

    print(
        f"Already processed: "
        f"{len(processed)}"
    )

    remaining = [
        pid
        for pid in product_ids
        if pid not in processed
    ]

    print(
        f"Remaining: "
        f"{len(remaining)}"
    )

    for idx, product_id in enumerate(
        remaining,
        start=1
    ):

        try:

            data = get_product(
                product_id
            )

            product = extract_product(
                data
            )

            save_jsonl(product)

            print(
                f"[{idx}/{len(remaining)}] "
                f"Saved "
                f"{product_id}"
            )

            time.sleep(0.5)

        except Exception as e:

            print(
                f"ERROR {product_id}: "
                f"{e}"
            )


if __name__ == "__main__":
    main()

Already processed: 0
Remaining: 1963
[1/1963] Saved 3992
[2/1963] Saved 4637
[3/1963] Saved 5636
[4/1963] Saved 6430
[5/1963] Saved 11877
[6/1963] Saved 11881
[7/1963] Saved 11984
[8/1963] Saved 11985
[9/1963] Saved 12102
[10/1963] Saved 14876
[11/1963] Saved 14997
[12/1963] Saved 14998
[13/1963] Saved 15095
[14/1963] Saved 20728
[15/1963] Saved 20780
[16/1963] Saved 20786
[17/1963] Saved 20791
[18/1963] Saved 20868
[19/1963] Saved 21071
[20/1963] Saved 21073
[21/1963] Saved 21080
[22/1963] Saved 21094
[23/1963] Saved 22881
[24/1963] Saved 23349
[25/1963] Saved 23506
[26/1963] Saved 23518
[27/1963] Saved 23525
[28/1963] Saved 23997
[29/1963] Saved 24113
[30/1963] Saved 26115
[31/1963] Saved 26116
[32/1963] Saved 26294
[33/1963] Saved 26297
[34/1963] Saved 36764
[35/1963] Saved 48011
[36/1963] Saved 49176
[37/1963] Saved 63529
[38/1963] Saved 64861
[39/1963] Saved 67299
[40/1963] Saved 92909
[41/1963] Saved 92910
[42/1963] Saved 93943
[43/1963] Saved 95059
[44/1963] Saved 95906
[45/1963

In [23]:
import json
from pathlib import Path

INPUT_FILE = "digikala_data/products.jsonl"

OUTPUT_DIR = Path("mobile_docs")
OUTPUT_DIR.mkdir(exist_ok=True)

def to_markdown(product):
    md = []
    md.append(f"# {product['title']}")
    md.append("")

    if product.get("brand"):
        md.append(f"**برند:** {product['brand']}")
        md.append("")

    if product.get("price"):
        md.append(f"**قیمت:** {product['price']}")
        md.append("")

    specs = product.get("specifications", {})
    for section_name, attrs in specs.items():

        md.append(f"## {section_name}")
        md.append("")

        for key, value in attrs.items():
            md.append(f"- **{key}:** {value}")

        md.append("")

    return "\n".join(md)

with open(INPUT_FILE, "r", encoding="utf-8") as f:

    for line in f:

        product = json.loads(line)
        product_id = product["id"]
        markdown = to_markdown(product)

        with open(
            OUTPUT_DIR / f"{product_id}.md",
            "w",
            encoding="utf-8"
        ) as out:

            out.write(markdown)

print("Done")

Done


In [22]:
from pathlib import Path


DOCS_DIR = Path("mobile_docs")

def load_markdown_documents():
    documents = []

    for file_path in DOCS_DIR.glob("*.md"):
        text = file_path.read_text(encoding="utf-8")

        documents.append({
            "text": text,
            "source": file_path.name,
        })

    return documents

if __name__ == "__main__":
    documents = load_markdown_documents()

    print(f"Number of documents: {len(documents)}")

    for doc in documents[:3]:
        print("=" * 80)
        print("SOURCE:", doc["source"])
        print(doc["text"][:500])

Number of documents: 0


In [25]:
import re

def clean_markdown(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+$", "", text, flags=re.MULTILINE)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(
        r"^(#{1,6})\s+",
        r"\1 ",
        text,
        flags=re.MULTILINE
    )
    text = re.sub(r"\s*\|\s*", " | ", text)
    text = re.sub(r" +([،,:;])", r"\1", text)
    text = text.strip()

    return text

In [27]:
MAX_CHARS = 2500

def extract_product_info(text: str):
    """
    استخراج اطلاعات کلی محصول از ابتدای Markdown
    """

    product_title = None
    brand = None
    model = None

    # عنوان محصول
    title_match = re.search(
        r"^#\s+(.+)$",
        text,
        flags=re.MULTILINE
    )

    if title_match:
        product_title = title_match.group(1).strip()

    # برند
    brand_match = re.search(
        r"\*\*برند:\*\*\s*(.+)",
        text
    )

    if brand_match:
        brand = brand_match.group(1).strip()

    # مدل
    model_match = re.search(
        r"-\s*\*\*مدل:\*\*\s*(.+)",
        text
    )

    if model_match:
        model = model_match.group(1).strip()

    return {
        "product_title": product_title,
        "brand": brand,
        "model": model
    }


def split_large_text(text: str, max_chars: int = MAX_CHARS):
    """
    اگر یک section خیلی بزرگ باشد،
    آن را بر اساس پاراگراف‌ها به چند chunk تقسیم می‌کند.
    """

    if len(text) <= max_chars:
        return [text]

    paragraphs = re.split(r"\n\n+", text)

    chunks = []
    current = ""

    for paragraph in paragraphs:
        paragraph = paragraph.strip()

        if not paragraph:
            continue

        # اگر پاراگراف به تنهایی از max_chars بزرگ‌تر باشد
        if len(paragraph) > max_chars:

            if current:
                chunks.append(current)
                current = ""

            # تقسیم پاراگراف بزرگ
            for i in range(0, len(paragraph), max_chars):
                chunks.append(
                    paragraph[i:i + max_chars]
                )

            continue

        # اضافه کردن پاراگراف به chunk فعلی
        if len(current) + len(paragraph) + 2 <= max_chars:

            if current:
                current += "\n\n"

            current += paragraph

        else:
            chunks.append(current)
            current = paragraph

    if current:
        chunks.append(current)

    return chunks


def chunk_markdown(text: str, source: str):
    """
    تبدیل یک فایل Markdown به مجموعه‌ای از chunkها
    همراه با metadata.
    """

    # -------------------------
    # Cleaning
    # -------------------------

    text = clean_markdown(text)

    # -------------------------
    # Extract product info
    # -------------------------

    product_info = extract_product_info(text)

    product_title = product_info["product_title"]
    brand = product_info["brand"]
    model = product_info["model"]

    # -------------------------
    # Product ID
    # -------------------------

    product_id = source.rsplit(".", 1)[0]

    # -------------------------
    # Split by ## sections
    # -------------------------

    sections = re.split(
        r"(?=^## )",
        text,
        flags=re.MULTILINE
    )

    chunks = []

    for section in sections:

        section = section.strip()

        if not section:
            continue

        # -------------------------
        # Section title
        # -------------------------

        match = re.match(
            r"^## (.+)",
            section
        )

        if match:
            section_title = match.group(1).strip()
        else:
            section_title = "اطلاعات کلی"

        # -------------------------
        # Split large sections
        # -------------------------

        sub_chunks = split_large_text(section)

        for chunk_index, sub_chunk in enumerate(sub_chunks):

            # -------------------------
            # Add product context
            # -------------------------

            contextual_text = (
                f"محصول: {product_title}\n"
                f"برند: {brand}\n"
                f"بخش: {section_title}\n\n"
                f"{sub_chunk}"
            )

            # -------------------------
            # Metadata
            # -------------------------

            metadata = {
                "source": source,
                "product_id": product_id,
                "product_title": product_title,
                "brand": brand,
                "model": model,
                "section": section_title,
                "chunk_index": chunk_index
            }

            chunks.append({
                "text": contextual_text,
                "metadata": metadata
            })

    return chunks

In [28]:
from pathlib import Path
import json


INPUT_DIR = Path("mobile_docs")
OUTPUT_DIR = Path("chunks")

OUTPUT_DIR.mkdir(exist_ok=True)

all_chunks = []
files = list(INPUT_DIR.glob("*.md"))
print(f"Found {len(files)} markdown files")

for index, file_path in enumerate(files, start=1):

    try:
        text = file_path.read_text(
            encoding="utf-8"
        )

        chunks = chunk_markdown(
            text=text,
            source=file_path.name
        )

        all_chunks.extend(chunks)
        output_file = OUTPUT_DIR / f"{file_path.stem}.json"

        output_file.write_text(
            json.dumps(
                chunks,
                ensure_ascii=False,
                indent=2
            ),
            encoding="utf-8"
        )

        print(
            f"[{index}/{len(files)}] "
            f"{file_path.name} → "
            f"{len(chunks)} chunks"
        )

    except Exception as e:

        print(
            f"ERROR: {file_path.name}"
        )

        print(e)

all_chunks_file = OUTPUT_DIR / "all_chunks.json"
all_chunks_file.write_text(
    json.dumps(
        all_chunks,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print("\n" + "=" * 60)
print("DONE")
print("=" * 60)

print(f"Products: {len(files)}")
print(f"Total chunks: {len(all_chunks)}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Combined file: {all_chunks_file}")

Found 1963 markdown files
[1/1963] 10002298.md → 10 chunks
[2/1963] 10012967.md → 10 chunks
[3/1963] 10013409.md → 10 chunks
[4/1963] 10013468.md → 10 chunks
[5/1963] 100400.md → 9 chunks
[6/1963] 10044754.md → 10 chunks
[7/1963] 10053559.md → 10 chunks
[8/1963] 1006836.md → 9 chunks
[9/1963] 101430.md → 10 chunks
[10/1963] 10177280.md → 10 chunks
[11/1963] 10182771.md → 10 chunks
[12/1963] 10227845.md → 10 chunks
[13/1963] 10251419.md → 10 chunks
[14/1963] 10259664.md → 10 chunks
[15/1963] 10260369.md → 10 chunks
[16/1963] 10261550.md → 11 chunks
[17/1963] 10310384.md → 10 chunks
[18/1963] 10310708.md → 10 chunks
[19/1963] 10311245.md → 10 chunks
[20/1963] 103153.md → 10 chunks
[21/1963] 10319905.md → 10 chunks
[22/1963] 103285.md → 10 chunks
[23/1963] 10340639.md → 10 chunks
[24/1963] 10374864.md → 10 chunks
[25/1963] 10375296.md → 10 chunks
[26/1963] 10383569.md → 10 chunks
[27/1963] 10402620.md → 10 chunks
[28/1963] 10403039.md → 10 chunks
[29/1963] 10403200.md → 10 chunks
[30/1963

In [1]:
!pip install tiktoken

# Making embeddings from chunks

In [4]:
import json
import time
import os
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv

INPUT_FILE = "chunks/all_chunks.json"
OUTPUT_FILE = "chunks/embedded_chunks.json"

MODEL = "text-embedding-3-large"
BATCH_SIZE = 1000  

load_dotenv(override=True)
gap_api_key = os.getenv('GAP_FERMIN_KEY')
gapgpt = OpenAI(api_key=gap_api_key, base_url="https://api.gapgpt.app/v1")

print("Loading chunks...")
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    chunks = json.load(f)

total_batches = (len(chunks) + BATCH_SIZE - 1) // BATCH_SIZE
print(f"Total chunks: {len(chunks):,} | Total batches: {total_batches}")

embedded_chunks = []

for i in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[i : i + BATCH_SIZE]
    
    texts = [chunk["text"] for chunk in batch]
    batch_num = (i // BATCH_SIZE) + 1
    
    print(f"Processing batch {batch_num}/{total_batches} ...")
    
    try:
        response = gapgpt.embeddings.create(
            input=texts,
            model=MODEL
        )
        
        for j, data in enumerate(response.data):
            chunk_copy = batch[j].copy()
            chunk_copy["embedding"] = data.embedding
            embedded_chunks.append(chunk_copy)
            
        time.sleep(0.5)
        
    except Exception as e:
        print(f"Error in batch {batch_num}: {e}")

print("Saving embedded chunks...")
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(embedded_chunks, f, ensure_ascii=False, indent=2)

print(f"Done! {len(embedded_chunks)} chunks successfully embedded and saved to {OUTPUT_FILE}")

Loading chunks...
Total chunks: 19,453 | Total batches: 20
Processing batch 1/20 ...
Processing batch 2/20 ...
Processing batch 3/20 ...
Processing batch 4/20 ...
Processing batch 5/20 ...
Processing batch 6/20 ...
Processing batch 7/20 ...
Processing batch 8/20 ...
Processing batch 9/20 ...
Processing batch 10/20 ...
Processing batch 11/20 ...
Processing batch 12/20 ...
Processing batch 13/20 ...
Processing batch 14/20 ...
❌ Error in batch 14: Error code: 403 - {'error': {'message': 'pre-consume quota failed, remaining user quota: ＄0.014794, required pre-consume quota: ＄0.018812 (request id: 202609071943473210448378268d9d6LqK9x2mN)', 'type': 'gap_api_error', 'param': '', 'code': 'insufficient_user_quota'}}
Processing batch 15/20 ...
❌ Error in batch 15: Error code: 403 - {'error': {'message': 'pre-consume quota failed, remaining user quota: ＄0.014794, required pre-consume quota: ＄0.020288 (request id: 202609071943484064450868268d9d6RGEy16os)', 'type': 'gap_api_error', 'param': '', 'co

## making embeddings for the remaining chunks

In [ ]:
import json
import time
from openai import OpenAI

INPUT_FILE = "chunks/all_chunks.json"
PARTIAL_OUTPUT_FILE = "chunks/embedded_chunks.json"
FINAL_OUTPUT_FILE = "chunks/embedded_chunks_complete.json"

MODEL = "text-embedding-3-large"
BATCH_SIZE = 1000

load_dotenv(override=True)
gap_api_key = os.getenv('GAP_FERMIN_KEY')
gapgpt = OpenAI(api_key=gap_api_key, base_url="https://api.gapgpt.app/v1")

print("Loading files...")
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    all_chunks = json.load(f)

with open(PARTIAL_OUTPUT_FILE, "r", encoding="utf-8") as f:
    embedded_chunks = json.load(f)

embedded_texts = {chunk["text"] for chunk in embedded_chunks}

missing_chunks = [chunk for chunk in all_chunks if chunk["text"] not in embedded_texts]

print("="*40)
print(f"Total original chunks: {len(all_chunks):,}")
print(f"Already embedded:      {len(embedded_chunks):,}")
print(f"Missing chunks:        {len(missing_chunks):,}")
print("="*40)

if missing_chunks:
    total_batches = (len(missing_chunks) + BATCH_SIZE - 1) // BATCH_SIZE
    newly_embedded = []

    for i in range(0, len(missing_chunks), BATCH_SIZE):
        batch = missing_chunks[i : i + BATCH_SIZE]
        texts = [chunk["text"] for chunk in batch]
        batch_num = (i // BATCH_SIZE) + 1
        
        print(f"Processing missing batch {batch_num}/{total_batches} ...")
        
        try:
            response = gapgpt.embeddings.create(
                input=texts,
                model=MODEL
            )
            
            for j, data in enumerate(response.data):
                chunk_copy = batch[j].copy()
                chunk_copy["embedding"] = data.embedding
                newly_embedded.append(chunk_copy)
                
            time.sleep(0.5)
            
        except Exception as e:
            print(f"Error in batch {batch_num}: {e}")
            break

    if len(newly_embedded) == len(missing_chunks):
        all_embedded_chunks = embedded_chunks + newly_embedded
        print(f"\nSaving {len(all_embedded_chunks):,} total embedded chunks...")
        
        with open(FINAL_OUTPUT_FILE, "w", encoding="utf-8") as f:
            json.dump(all_embedded_chunks, f, ensure_ascii=False, indent=2)
            
        print(f"Fixed successfully! File saved to: {FINAL_OUTPUT_FILE}")
    else:
        print("\nProcess interrupted again. Please check your API quota or network.")

else:
    print("No missing chunks found. Your file is already complete!")

Loading files...
Total original chunks: 19,453
Already embedded:      16,453
Missing chunks:        2,963
Processing missing batch 1/3 ...
Processing missing batch 2/3 ...
Processing missing batch 3/3 ...

Saving 19,416 total embedded chunks...
Fixed successfully! File saved to: chunks/embedded_chunks_complete.json


# Visualizing embeddings (3D)

In [8]:
import numpy as np
import pandas as pd
from sklearn.manifold import TSNE
import plotly.express as px
from chromadb import PersistentClient
from collections import Counter

DB_NAME = "digikala_chroma_db"
COLLECTION_NAME = "mobiles"
SAMPLE_SIZE = 3000 

print("Loading vectors from ChromaDB...")
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_collection(COLLECTION_NAME)

result = collection.get(include=['embeddings', 'documents', 'metadatas'])

total_docs = len(result['embeddings'])
print(f"Total documents in DB: {total_docs:,}")

np.random.seed(42)
indices = np.random.choice(
    total_docs, 
    min(SAMPLE_SIZE, total_docs), 
    replace=False
)

vectors = np.array(result['embeddings'])[indices]
documents = [result['documents'][i] for i in indices]
metadatas = [result['metadatas'][i] for i in indices]

all_brands = [m.get('brand', 'نامشخص') for m in metadatas]
all_brands = [b if b and b.strip() else 'نامشخص' for b in all_brands]

top_brands = [brand for brand, count in Counter(all_brands).most_common(7)]

categories = [b if b in top_brands else 'سایر' for b in all_brands]

print(f"Running t-SNE on {len(vectors)} vectors... (This may take a minute)")
tsne = TSNE(n_components=3, random_state=42, perplexity=30)
reduced_vectors = tsne.fit_transform(vectors)

print("Plotting...")
df = pd.DataFrame({
    'x': reduced_vectors[:, 0],
    'y': reduced_vectors[:, 1],
    'z': reduced_vectors[:, 2],
    'Brand': categories,
    'Text': [d[:150] + "..." for d in documents] 
})

fig = px.scatter_3d(
    df, x='x', y='y', z='z',
    color='Brand',
    hover_name='Brand',
    hover_data={'Text': True, 'x': False, 'y': False, 'z': False},
    title=f'3D Chroma Vector Store Visualization ({len(vectors)} Sampled Mobile Chunks)'
)

fig.update_traces(marker=dict(size=4, opacity=0.75))
fig.update_layout(
    width=1000, 
    height=800,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

Loading vectors from ChromaDB...
Total documents in DB: 19,416
Running t-SNE on 3000 vectors... (This may take a minute)
Plotting...


# Saving embeddings in vectorstore

In [7]:
import json
import chromadb
from chromadb import PersistentClient

INPUT_FILE = "chunks/embedded_chunks_complete.json"  
DB_NAME = "digikala_chroma_db"
COLLECTION_NAME = "mobiles"

print("Loading embedded chunks from file...")
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    chunks = json.load(f)

chroma = PersistentClient(path=DB_NAME)

existing_collections = [c.name for c in chroma.list_collections()]
if COLLECTION_NAME in existing_collections:
    print(f"Deleting existing collection '{COLLECTION_NAME}'...")
    chroma.delete_collection(COLLECTION_NAME)

collection = chroma.get_or_create_collection(COLLECTION_NAME)

ids = []
embeddings = []
documents = []
metadatas = []

for i, chunk in enumerate(chunks):
    ids.append(str(i)) 
    embeddings.append(chunk["embedding"]) 
    documents.append(chunk["text"])
    
    clean_meta = {}
    for key, value in chunk["metadata"].items():
        if value is not None:
            clean_meta[key] = value
        else:
            clean_meta[key] = "نامشخص" 
            
    metadatas.append(clean_meta)

BATCH_SIZE = 5000 

print("Inserting data into ChromaDB...")
for i in range(0, len(ids), BATCH_SIZE):
    collection.add(
        ids=ids[i : i + BATCH_SIZE],
        embeddings=embeddings[i : i + BATCH_SIZE],
        documents=documents[i : i + BATCH_SIZE],
        metadatas=metadatas[i : i + BATCH_SIZE]
    )
    batch_num = (i // BATCH_SIZE) + 1
    print(f"Batch {batch_num} inserted successfully.")

print(f"Vectorstore created with {collection.count()} documents")

Loading embedded chunks from file...
Deleting existing collection 'mobiles'...
Inserting data into ChromaDB...
Batch 1 inserted successfully.
Batch 2 inserted successfully.
Batch 3 inserted successfully.
Batch 4 inserted successfully.
Vectorstore created with 19416 documents


# Advanced RAG techniques and making the responder

In [10]:
from openai import OpenAI
import chromadb
from pydantic import BaseModel, Field
from dataclasses import dataclass
from typing import List

DB_NAME = "digikala_chroma_db"
COLLECTION_NAME = "mobiles"
EMBEDDING_MODEL = "text-embedding-3-large"
MODEL = "gpt-4o-mini"

chroma_client = chromadb.PersistentClient(path=DB_NAME)
collection = chroma_client.get_collection(COLLECTION_NAME)

@dataclass
class Result:
    page_content: str
    metadata: dict

class RankOrder(BaseModel):
    order: List[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )

In [11]:
def rewrite_query(question: str, history: list[dict] = []) -> str:
    """
    Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base.
    """

    message = f"""
You are a smart search assistant for Navikala's mobile phone database.
Based on the following conversation history and the user's current question, generate a highly precise, short, and independent search query.
This query will be used for semantic (vector) search in the database. Focus on features, brands, budget, and models.

Conversation History:
{history}

Current User Question:
{question}

IMPORTANT: Return ONLY the final search query and absolutely nothing else.
"""
    response = gapgpt.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": message}],
        temperature=0.2
    )
    
    rewritten_query = response.choices[0].message.content.strip()
    return rewritten_query

In [17]:
RETRIEVAL_K = 15

def fetch_context_unranked(question: str) -> List[Result]:
    query_emb = gapgpt.embeddings.create(
        model=EMBEDDING_MODEL, 
        input=[question]
    ).data[0].embedding
    
    results = collection.query(
        query_embeddings=[query_emb], 
        n_results=RETRIEVAL_K
    )
    
    chunks = []
    if results["documents"] and len(results["documents"]) > 0:
        for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
            chunks.append(Result(page_content=doc, metadata=meta))
            
    return chunks

In [18]:
def rerank(question: str, chunks: List[Result]) -> List[Result]:
    if not chunks:
        return []
        
    system_prompt = """
You are an expert Document Re-ranker.
You are provided with a user's question and a list of text chunks retrieved from a database.
Your task is to rank these chunks from most relevant to least relevant based on their usefulness in answering the user's question.
Include the IDs of all provided chunks in your output.
"""
    
    user_prompt = f"User's Question:\n{question}\n\nRank the following documents by relevance:\n\n"
    
    for index, chunk in enumerate(chunks):
        user_prompt += f"--- CHUNK ID: {index + 1} ---\n{chunk.page_content}\n\n"
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    
    try:
        response = gapgpt.beta.chat.completions.parse(
            model=MODEL, 
            messages=messages, 
            response_format=RankOrder,
            temperature=0
        )
        
        order = response.choices[0].message.parsed.order
        
        valid_order = [i for i in order if 1 <= i <= len(chunks)]
        
        print(f"Reranked order: {valid_order}")
        return [chunks[i - 1] for i in valid_order]
        
    except Exception as e:
        print(f"Rerank failed: {e}. Returning original chunks.")
        return chunks

In [19]:
def fetch_context(question: str) -> List[Result]:
    unranked_chunks = fetch_context_unranked(question)
    ranked_chunks = rerank(question, unranked_chunks)
    
    return ranked_chunks[:8]

In [20]:
SYSTEM_PROMPT = """
You are a professional mobile phone expert and consultant at Navikala.
Your task is to provide accurate, polite, and honest guidance to users looking to buy a phone.
Use the following context (retrieved from the product database) to answer the user.

Retrieved Database Information:
{context}

Response Rules:
1. Answer strictly based on the provided information above.
2. If the answer is not contained in the provided information, explicitly state that you do not have enough information and do not guess.
3. Mention model names and key specifications when necessary.
4. Your final response MUST be in Persian (Farsi), maintaining a friendly and professional tone.
"""

def make_rag_messages(question: str, history: list[dict], chunks: List[Result]) -> list:
    context_parts = []
    for chunk in chunks:
        brand = chunk.metadata.get('brand', 'نامشخص')
        title = chunk.metadata.get('product_title', 'محصول نامشخص')
        
        context_parts.append(
            f"📱 [محصول: {title} | برند: {brand}]\n{chunk.page_content}"
        )
        
    context_str = "\n\n".join(context_parts)
    system_message = SYSTEM_PROMPT.format(context=context_str)
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": question}]
    return messages

In [21]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    print("1. Rewriting query...")
    query = rewrite_query(question, history)
    print(f"   Optimizied Query: {query}\n")
    
    print("2. Fetching and Reranking context...")
    top_chunks = fetch_context(query)
    
    print("\n3. Generating final answer...")
    messages = make_rag_messages(question, history, top_chunks)
    
    response = gapgpt.chat.completions.create(
        model=MODEL, 
        messages=messages,
        temperature=0.5
    )
    
    return response.choices[0].message.content, top_chunks

In [23]:
from IPython.display import display, Markdown

# تست پایپ‌لاین با یک سوال نمونه
user_question = "ارزان‌ترین گوشی سامسونگ که دوربینش حداقل ۵۰ مگاپیکسل باشه چیه؟"

# لاگ‌های انگلیسی اینجا چاپ می‌شوند (چپ‌چین)
answer, retrieved_chunks = answer_question(question=user_question, history=[])

# -----------------------------
# قالب‌بندی خروجی نهایی به صورت راست‌چین (RTL)
# -----------------------------

# ساخت ساختار مارک‌داون برای پاسخ
output_md = f"### 🤖 پاسخ بات:\n\n---\n\n{answer}\n\n---\n\n### 📚 سورس‌های استفاده شده:\n"

for i, chunk in enumerate(retrieved_chunks):
    title = chunk.metadata.get('product_title', 'نامشخص')
    brand = chunk.metadata.get('brand', 'نامشخص')
    
    # استفاده از تگ LTR برای پرانتز برند تا عددها و حروف انگلیسی به هم نریزند
    output_md += f"{i+1}. {title} <span dir='ltr' style='color:gray; font-size:0.9em;'>(Brand: {brand})</span>\n"

# رندر نهایی با پوشش HTML برای اجبار به راست‌چین شدن کل بلاک
display(Markdown(f"<div dir='rtl' style='text-align: right;'>\n\n{output_md}\n\n</div>"))

1. Rewriting query...
   Optimizied Query: ارزان‌ترین گوشی سامسونگ با دوربین ۵۰ مگاپیکسل

2. Fetching and Reranking context...
Reranked order: [2, 3, 10, 11, 5, 15, 1, 4, 6, 12, 13, 7, 8, 9]

3. Generating final answer...


<div dir='rtl' style='text-align: right;'>

### 🤖 پاسخ بات:

---

ارزان‌ترین گوشی سامسونگ که دارای دوربین اصلی با حداقل ۵۰ مگاپیکسل باشد، مدل **Galaxy A54 5G** است. این گوشی دارای دوربین اصلی با رزولوشن ۵۰ مگاپیکسل و همچنین دو دوربین دیگر با رزولوشن‌های ۱۲ و ۵ مگاپیکسل می‌باشد. 

اگر به اطلاعات بیشتری نیاز دارید یا سوال دیگری دارید، خوشحال می‌شوم کمک کنم!

---

### 📚 سورس‌های استفاده شده:
1. گوشی موبایل سامسونگ مدل Galaxy A55 دو سیم کارت ظرفیت 256 گیگابایت و رم 12 گیگابایت - اکتیو - به همراه شارژر 25 وات سامسونگ <span dir='ltr' style='color:gray; font-size:0.9em;'>(Brand: سامسونگ)</span>
2. گوشی موبایل سامسونگ مدل Galaxy A56 دو سیم کارت ظرفیت 256 گیگابایت و رم 8 گیگابایت - ویتنام - به همراه شارژر 45 واتی سامسونگ، هدفون بلوتوثی انکر مدل Soundcore R50i، محافظ صفحه نمایش و قاب ژله‌ای <span dir='ltr' style='color:gray; font-size:0.9em;'>(Brand: سامسونگ)</span>
3. گوشی موبایل سامسونگ مدل Galaxy A54 5G دو سیم کارت ظرفیت 256 گیگابایت و رم 8 گیگابایت - به همراه شارژر 25 وات سامسونگ <span dir='ltr' style='color:gray; font-size:0.9em;'>(Brand: سامسونگ)</span>
4. گوشی موبایل سامسونگ مدل Galaxy A54 5G دو سیم کارت ظرفیت 128 گیگابایت و رم 8 گیگابایت به همراه شارژر 25وات سامسونگ <span dir='ltr' style='color:gray; font-size:0.9em;'>(Brand: سامسونگ)</span>
5. گوشی موبایل سامسونگ مدل Galaxy A56 دو سیم کارت ظرفیت 256 گیگابایت و رم 12 گیگابایت - به همراه شارژر 25 وات سامسونگ <span dir='ltr' style='color:gray; font-size:0.9em;'>(Brand: سامسونگ)</span>
6. گوشی موبایل سامسونگ مدل Galaxy A55 دو سیم کارت ظرفیت 128 گیگابایت و رم 8 گیگابایت <span dir='ltr' style='color:gray; font-size:0.9em;'>(Brand: سامسونگ)</span>
7. گوشی موبایل سامسونگ مدل Galaxy S22 Ultra 5G دو سیم کارت ظرفیت 512 گیگابایت و رم 12 گیگابایت <span dir='ltr' style='color:gray; font-size:0.9em;'>(Brand: سامسونگ)</span>
8. گوشی موبایل سامسونگ مدل Galaxy S22 Ultra 5G دو سیم کارت ظرفیت 256 گیگابایت و رم 12 گیگابایت نسخه اسنپدراگون به همراه شارژر سامسونگ <span dir='ltr' style='color:gray; font-size:0.9em;'>(Brand: سامسونگ)</span>


</div>

# Making Gradio UI

In [ ]:
import gradio as gr

def chat_interface(user_message, history):
    """
    This function takes the user message and Gradio history,
    converts them into OpenAI format, and passes them to our RAG function.
    """
    
    # Convert Gradio history (list of lists) to OpenAI dictionary format
    formatted_history = []
    for user_text, bot_text in history:
        formatted_history.append({"role": "user", "content": user_text})
        # Remove HTML tags from history for cleaner prompting
        clean_bot_text = bot_text.replace("<div dir='rtl' style='text-align: right;'>", "").replace("</div>", "")
        formatted_history.append({"role": "assistant", "content": clean_bot_text})

    answer, chunks = answer_question(user_message, formatted_history)
    
    # -----------------------------
    # 2. Extract top product information
    # -----------------------------
    product_html = "<div dir='rtl' style='text-align:right; color:gray; padding:20px;'>No product found to display.</div>"
    
    if chunks:
        # Assuming the first chunk (most relevant) is the primary recommended product
        top_chunk = chunks[0]
        p_id = top_chunk.metadata.get('product_id', '')
        p_title = top_chunk.metadata.get('product_title', 'Unknown')
        p_brand = top_chunk.metadata.get('brand', 'Unknown')
        
        if p_id:
            dk_url = f"https://www.digikala.com/product/dkp-{p_id}/"
            
            product_html = f"""
            <div dir="rtl" style="text-align: right; padding: 20px; border: 2px solid #ef394e; border-radius: 12px; background-color: #fff; box-shadow: 0 4px 8px rgba(0,0,0,0.1);">
                <div style="text-align: center; font-size: 3rem; margin-bottom: 10px;">📱</div >
                <h4 style="margin-top:0; color: #ef394e; font-size: 1.1rem;">Special Offer:</h4>
                <p style="font-size: 0.95rem; font-weight: bold; color: #333; line-height: 1.5;">{p_title}</p>
                <p style="font-size: 0.85rem; color: #777; margin-bottom: 15px;">Brand: {p_brand}</p>
                <a href="{dk_url}" target="_blank" style="display: block; text-align: center; padding: 10px 15px; background-color: #ef394e; color: white; text-decoration: none; border-radius: 8px; font-weight: bold; transition: background 0.3s;">
                    🛒 View on Digikala
                </a>
            </div >
            """

    rtl_answer = f"<div dir='rtl' style='text-align: right; line-height: 1.8;'>{answer}</div>"
    
    history.append((user_message, rtl_answer))
    
    return "", history, product_html

# -----------------------------
# 3. UI Design (Graphical Interface)
# -----------------------------
custom_css = """
@import url('https://cdn.jsdelivr.net/gh/rastikerdar/vazirmatn@v33.003/Vazirmatn-font-face.css');
.gradio-container { font-family: 'Vazirmatn', sans-serif !important; }
.message-wrap { direction: rtl !important; }
textarea { direction: rtl !important; }
"""

with gr.Blocks(css=custom_css, theme=gr.themes.Soft(primary_hue="red")) as demo:
    gr.HTML("""
    <div style='text-align: center; padding: 10px; background-color: #ef394e; color: white; border-radius: 10px; margin-bottom: 20px;'>
        <h1 style='margin: 0; font-size: 1.8rem;'>🤖 Nivikala Smart Assistant (Mobile Expert)</h1>
        <p style='margin: 5px 0 0 0; opacity: 0.9;'>Shopping advice, comparison, and precise mobile searching</p>
    </div >
    """)
    
    with gr.Row():
        # Right Section (Sidebar for Product Card) - 1/4 of the screen
        with gr.Column(scale=1):
            product_display = gr.HTML(
                value="<div dir='rtl' style='text-align:right; color:gray; padding:20px; border: 1px dashed #ccc; border-radius: 10px;'>Ask a question to see a recommended product here...</div >",
                label="Recommended Product"
            )
            
        # Left Section (Chat Environment and Input) - 3/4 of the screen
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                label="Conversation", 
                height=500, 
                show_label=False,
                bubble_full_width=False
            )
            
            with gr.Row():
                msg = gr.Textbox(
                    scale=4,
                    show_label=False,
                    placeholder="e.g., What is the cheapest Samsung phone with a 5000mAh battery?",
                    container=False
                )
                submit_btn = gr.Button("Send", scale=1, variant="primary")

    # Event bindings (Trigger on 'Enter' in textbox or clicking the 'Send' button)
    msg.submit(
        chat_interface, 
        inputs=[msg, chatbot], 
        outputs=[msg, chatbot, product_display]
    )
    submit_btn.click(
        chat_interface, 
        inputs=[msg, chatbot], 
        outputs=[msg, chatbot, product_display]
    )

# -----------------------------
# 4. Execution
# -----------------------------
demo.launch(share=False, inbrowser=True)


C:\Users\USER\AppData\Local\Temp\ipykernel_18100\2967869425.py:80: UserWarning:

You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.

C:\Users\USER\AppData\Local\Temp\ipykernel_18100\2967869425.py:80: DeprecationWarning:

The 'bubble_full_width' parameter is deprecated and will be removed in a future version. This parameter no longer has any effect.



* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


# Uploading Markdowns and Embeddings in Hugging Face

In [ ]:
import os
import json
from pathlib import Path
import pandas as pd
from datasets import Dataset
from huggingface_hub import login
from dotenv import load_dotenv

load_dotenv(override=True)
gap_api_key = os.getenv('GAPGPT_API_KEY')

login(token="hf_YOUR_WRITE_TOKEN_HERE")

# نام کاربری و نام دلخواه برای دیتاست را مشخص کنید
REPO_ID = "YOUR_USERNAME/digikala-mobile-expert-data"


# ==========================================
# بخش اول: پردازش و آپلود فایل‌های مارک‌داون
# ==========================================
print("📦 Processing raw Markdown files...")
MARKDOWN_DIR = Path("mobile_docs")

md_data = []
# خواندن تمام فایل‌های مارک‌داون از پوشه
for file_path in MARKDOWN_DIR.glob("*.md"):
    md_data.append({
        "product_id": file_path.stem, # استخراج آیدی از نام فایل (بدون .md)
        "markdown_text": file_path.read_text(encoding="utf-8")
    })

md_df = pd.DataFrame(md_data)
md_dataset = Dataset.from_pandas(md_df)

print("🚀 Pushing Markdowns to Hugging Face...")
# آپلود تحت نام پیکربندی 'markdowns'
md_dataset.push_to_hub(REPO_ID, config_name="markdowns", split="train")


# ==========================================
# بخش دوم: پردازش و آپلود چانک‌ها و امبدینگ‌ها
# ==========================================
print("\n📦 Processing embedded chunks...")
EMBEDDINGS_FILE = "chunks/embedded_chunks_complete.json"

with open(EMBEDDINGS_FILE, "r", encoding="utf-8") as f:
    chunks_data = json.load(f)

flat_chunks = []
for item in chunks_data:
    flat_chunks.append({
        "text": item["text"],
        "embedding": item["embedding"],
        "product_id": item["metadata"].get("product_id"),
        "product_title": item["metadata"].get("product_title"),
        "brand": item["metadata"].get("brand", "نامشخص"),
        "section": item["metadata"].get("section")
    })

emb_df = pd.DataFrame(flat_chunks)
emb_dataset = Dataset.from_pandas(emb_df)

print("🚀 Pushing Embeddings to Hugging Face...")
# آپلود تحت نام پیکربندی 'embeddings'
emb_dataset.push_to_hub(REPO_ID, config_name="embeddings", split="train")

print("\n✅ All done! Datasets are successfully uploaded and separated by configs.")